# 📊 Market Mix Modelling — Complete Pipeline with Bayesian Optimization

> **A 12-step, production-grade MMM framework** with Bayesian-optimized adstock and saturation parameters via Optuna TPE.

---

| Detail | Value |
|:---|:---|
| **Dataset** | 156 weeks (Jan 2021 – Dec 2023), synthetic FMCG brand |
| **Media Channels** | TV, Digital, Print, Radio, OOH |
| **Controls** | Price Index, Distribution %, Competitor Spend, Promotions |
| **Method** | OLS Regression with Adstock + Hill Saturation |
| **Optimization** | Optuna TPE (Tree-structured Parzen Estimator) for joint parameter calibration |

### Pipeline Overview

| Step | Description |
|:---:|:---|
| 1 | Data Loading & Exploratory Analysis |
| 2 | Correlation & Multicollinearity (VIF) |
| 3 | Adstock Transformation — Manual Baseline |
| 4 | Saturation Curves — Manual Baseline |
| 5 | **Bayesian Optimization (Optuna TPE)** — Joint calibration of decay, K, alpha |
| 6 | Feature Engineering (Seasonality, Dummies) |
| 7 | Model Building — Manual vs Bayesian (OLS + CV) |
| 8 | Model Diagnostics (Residuals, Statistical Tests) |
| 9 | Contribution Decomposition & Waterfall |
| 10 | ROI / Effectiveness Analysis |
| 11 | Budget Optimization (Constrained SLSQP) |
| 12 | Final Comparison & Recommendations |

---
## ⚙️ Step 0 — Environment Setup

In [ ]:
!pip install -q statsmodels scikit-learn seaborn scipy openpyxl optuna

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import optuna
from scipy.optimize import minimize
from scipy.stats import norm
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan, acorr_breusch_godfrey
from matplotlib.patches import Patch

optuna.logging.set_verbosity(optuna.logging.WARNING)

%matplotlib inline
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({
    'figure.figsize': (14, 6),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.dpi': 120,
})

print("All libraries loaded successfully.")

---
### Generate the Synthetic MMM Dataset

The generator encodes **known ground-truth coefficients** with realistic dynamics: adstock decay, Hill saturation, seasonal spikes, price elasticity, and competitor dampening.

In [ ]:
np.random.seed(42)

N_WEEKS = 156
START_DATE = "2021-01-04"
dates = pd.date_range(start=START_DATE, periods=N_WEEKS, freq="W-MON")
week_of_year = dates.isocalendar().week.values.astype(float)
month = dates.month

# Media Spends (INR Lakhs)
tv_spend = np.random.lognormal(mean=4.5, sigma=0.5, size=N_WEEKS).clip(20, 300)
festive_mask = np.isin(month, [9, 10, 11, 12])
tv_spend[festive_mask] *= np.random.uniform(1.3, 1.8, size=festive_mask.sum())

digital_spend = np.random.lognormal(mean=3.8, sigma=0.4, size=N_WEEKS).clip(10, 200)
digital_spend *= np.linspace(1.0, 1.8, N_WEEKS)

print_spend = np.random.lognormal(mean=3.0, sigma=0.6, size=N_WEEKS).clip(5, 100)
radio_spend = np.random.lognormal(mean=2.8, sigma=0.5, size=N_WEEKS).clip(3, 80)
ooh_spend = np.random.lognormal(mean=3.2, sigma=0.4, size=N_WEEKS).clip(8, 120)

# Controls
base_price = 100
price_index = (base_price + np.cumsum(np.random.normal(0, 0.5, N_WEEKS))).clip(85, 118)
distribution = (70 + np.cumsum(np.random.normal(0.05, 0.3, N_WEEKS))).clip(60, 95)
competitor_spend = np.random.lognormal(mean=5.0, sigma=0.3, size=N_WEEKS).clip(50, 400)
promo_flag = np.random.binomial(1, 0.25, N_WEEKS)
promo_depth = promo_flag * np.random.uniform(5, 25, N_WEEKS)

# Helpers for data generation
def _adstock(x, decay):
    out = np.zeros_like(x, dtype=float)
    out[0] = x[0]
    for i in range(1, len(x)):
        out[i] = x[i] + decay * out[i - 1]
    return out

def _hill(x, K, s):
    xn = x / x.max()
    return xn**s / (xn**s + K**s)

# True transforms used to generate data
tv_sat      = _hill(_adstock(tv_spend, 0.70), 0.50, 1.8)
digital_sat = _hill(_adstock(digital_spend, 0.40), 0.40, 2.2)
print_sat   = _hill(_adstock(print_spend, 0.50), 0.50, 1.5)
radio_sat   = _hill(_adstock(radio_spend, 0.30), 0.45, 1.6)
ooh_sat     = _hill(_adstock(ooh_spend, 0.55), 0.50, 1.7)

# Seasonality
seasonality = (
    0.08 * np.sin(2 * np.pi * week_of_year / 52)
    + 0.05 * np.sin(4 * np.pi * week_of_year / 52)
    + 0.12 * np.where(np.isin(week_of_year, [40,41,42,43,44,45]), 1, 0)
    + 0.06 * np.where(np.isin(month, [12, 1]), 1, 0)
)

# Revenue with known coefficients
revenue = (
    800 + 320*tv_sat + 250*digital_sat + 80*print_sat + 55*radio_sat + 110*ooh_sat
    - 4.5*(price_index - base_price) + 6.0*(distribution - 70)
    - 0.3*(competitor_spend / 100) + 3.8*promo_depth
)
revenue *= (1 + seasonality)
revenue += np.random.normal(0, revenue.mean() * 0.05, N_WEEKS)
revenue = revenue.clip(min=300)

df = pd.DataFrame({
    'date': dates, 'week_number': np.arange(1, N_WEEKS + 1),
    'revenue_lakhs': np.round(revenue, 2),
    'tv_spend_lakhs': np.round(tv_spend, 2),
    'digital_spend_lakhs': np.round(digital_spend, 2),
    'print_spend_lakhs': np.round(print_spend, 2),
    'radio_spend_lakhs': np.round(radio_spend, 2),
    'ooh_spend_lakhs': np.round(ooh_spend, 2),
    'price_index': np.round(price_index, 2),
    'distribution_pct': np.round(distribution, 2),
    'competitor_spend_lakhs': np.round(competitor_spend, 2),
    'promo_flag': promo_flag,
    'promo_depth_pct': np.round(promo_depth, 2),
})
df.set_index('date', inplace=True)
df.to_csv('mmm_dataset.csv')

# Global constants used throughout
media_cols = ['tv_spend_lakhs', 'digital_spend_lakhs', 'print_spend_lakhs',
              'radio_spend_lakhs', 'ooh_spend_lakhs']
channel_names = ['TV', 'DIGITAL', 'PRINT', 'RADIO', 'OOH']
channel_keys = ['tv', 'digital', 'print', 'radio', 'ooh']
colors_media = ['#E74C3C', '#3498DB', '#27AE60', '#F39C12', '#9B59B6']

print(f"Dataset: {df.shape[0]} weeks x {df.shape[1]} columns")
print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")
print(f"Revenue: {df['revenue_lakhs'].min():.0f}L - {df['revenue_lakhs'].max():.0f}L (mean: {df['revenue_lakhs'].mean():.0f}L)")

---
## 📈 Step 1 — Exploratory Data Analysis

In [ ]:
df.describe().round(2)

In [ ]:
# Revenue time series
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df.index, df['revenue_lakhs'], color='#2C3E50', linewidth=1.5, label='Weekly Revenue')
ax.fill_between(df.index, df['revenue_lakhs'], alpha=0.15, color='#3498DB')
ax.axhline(df['revenue_lakhs'].mean(), color='#E74C3C', linestyle='--', alpha=0.7,
           label=f'Mean: {df["revenue_lakhs"].mean():.0f}L')
ax.set_title('Weekly Revenue Over 3 Years', fontweight='bold')
ax.set_ylabel('Revenue (Lakhs)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Media spend trends + budget split pie
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, (col, color) in enumerate(zip(media_cols, colors_media)):
    axes[i].plot(df.index, df[col], color=color, linewidth=1, alpha=0.8)
    axes[i].fill_between(df.index, df[col], alpha=0.15, color=color)
    axes[i].set_title(col.replace('_', ' ').title(), fontweight='bold')
    axes[i].set_ylabel('Lakhs')
    axes[i].tick_params(axis='x', rotation=30)

total_spends = df[media_cols].sum()
axes[5].pie(total_spends, labels=[c.split('_')[0].upper() for c in media_cols],
            colors=colors_media, autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11})
axes[5].set_title('Total Media Budget Split', fontweight='bold')
plt.suptitle('Media Spend Patterns', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 🔗 Step 2 — Correlation & Multicollinearity Analysis

In [ ]:
numeric_cols = ['revenue_lakhs'] + media_cols + ['price_index', 'distribution_pct',
                'competitor_spend_lakhs', 'promo_depth_pct']
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax,
            xticklabels=[c.replace('_lakhs','').replace('_',' ').title() for c in numeric_cols],
            yticklabels=[c.replace('_lakhs','').replace('_',' ').title() for c in numeric_cols])
ax.set_title('Correlation Matrix', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# VIF Analysis
X_vif = df[media_cols + ['price_index', 'distribution_pct', 'competitor_spend_lakhs', 'promo_depth_pct']]
X_vif_const = sm.add_constant(X_vif)
vif_data = pd.DataFrame({
    'Feature': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif_const.values, i+1) for i in range(X_vif.shape[1])]
}).sort_values('VIF', ascending=False).reset_index(drop=True)

print("Variance Inflation Factor (VIF) - Raw Features")
print("-" * 45)
print(vif_data.to_string(index=False))

---
## 📡 Step 3 — Adstock Transformation (Manual Baseline)

The geometric adstock model captures advertising carryover:

$$\text{Adstock}(t) = \text{Spend}(t) + \lambda \cdot \text{Adstock}(t-1)$$

We first establish a **manual baseline** using grid search for decay only, then improve it with Bayesian optimization in Step 5.

In [ ]:
def apply_adstock(series, decay):
    vals = series.values if hasattr(series, 'values') else np.asarray(series)
    out = np.zeros(len(vals), dtype=float)
    out[0] = vals[0]
    for t in range(1, len(vals)):
        out[t] = vals[t] + decay * out[t - 1]
    return out

def hill_transform(x, K, alpha):
    x_max = x.max()
    if x_max == 0:
        return x
    x_norm = x / x_max
    return x_norm ** alpha / (x_norm ** alpha + K ** alpha)

# Manual baseline parameters (grid-searched decay + hardcoded K, alpha)
MANUAL_PARAMS = {
    'tv':      {'decay': 0.70, 'K': 0.50, 'alpha': 1.8},
    'digital': {'decay': 0.40, 'K': 0.40, 'alpha': 2.2},
    'print':   {'decay': 0.50, 'K': 0.50, 'alpha': 1.5},
    'radio':   {'decay': 0.30, 'K': 0.45, 'alpha': 1.6},
    'ooh':     {'decay': 0.55, 'K': 0.50, 'alpha': 1.7},
}

# Grid search for decay (baseline method)
def find_best_decay(df, spend_col, target_col='revenue_lakhs',
                    decay_range=np.arange(0.1, 0.95, 0.05)):
    best_decay, best_corr = 0, 0
    results = []
    for d in decay_range:
        adstocked = apply_adstock(df[spend_col], d)
        corr = np.corrcoef(adstocked, df[target_col])[0, 1]
        results.append((d, corr))
        if abs(corr) > abs(best_corr):
            best_corr, best_decay = corr, d
    return best_decay, best_corr, results

print("Grid Search for Decay Rate (manual baseline)")
print("-" * 60)
decay_search_results = {}
for col, ch in zip(media_cols, channel_keys):
    best_d, best_c, grid = find_best_decay(df, col)
    decay_search_results[col] = grid
    print(f"  {col:30s} -> decay = {best_d:.2f}  (r = {best_c:.3f})")

In [ ]:
# Decay search curves
fig, axes = plt.subplots(1, 5, figsize=(22, 4), sharey=True)
for i, (col, color) in enumerate(zip(media_cols, colors_media)):
    decays, corrs = zip(*decay_search_results[col])
    axes[i].plot(decays, corrs, color=color, linewidth=2, marker='o', markersize=4)
    axes[i].axvline(MANUAL_PARAMS[channel_keys[i]]['decay'], color='black', linestyle='--', alpha=0.5)
    axes[i].set_title(col.split('_')[0].upper(), fontweight='bold')
    axes[i].set_xlabel('Decay Rate')
    if i == 0:
        axes[i].set_ylabel('Correlation with Revenue')
plt.suptitle('Adstock Decay Rate Search (Grid — Baseline)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 📉 Step 4 — Saturation Curves (Manual Baseline)

The Hill function models diminishing returns:

$$S(x) = \frac{x^{\alpha}}{x^{\alpha} + K^{\alpha}}$$

In the manual baseline, K and alpha are **hardcoded** based on domain heuristics. Step 5 will optimize them jointly with decay.

In [ ]:
# Plot manual saturation curves
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
x = np.linspace(0.001, 1, 200)
for i, (ch, color) in enumerate(zip(channel_keys, colors_media)):
    p = MANUAL_PARAMS[ch]
    y = x ** p['alpha'] / (x ** p['alpha'] + p['K'] ** p['alpha'])
    axes[i].plot(x, y, color=color, linewidth=2.5)
    axes[i].fill_between(x, y, alpha=0.1, color=color)
    axes[i].set_title(f"{ch.upper()}\nK={p['K']}, a={p['alpha']}", fontweight='bold')
    axes[i].set_xlabel('Normalized Spend')
    if i == 0:
        axes[i].set_ylabel('Saturated Effect')
    axes[i].axhline(0.5, linestyle=':', color='gray', alpha=0.5)
    axes[i].axvline(p['K'], linestyle=':', color='gray', alpha=0.5)
plt.suptitle('Hill Saturation Curves (Manual Baseline)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🧠 Step 5 — Bayesian Optimization (Optuna TPE)

### Why Bayesian over Grid Search?

The manual approach had two problems:
1. **Grid search for decay only** — tested 17 values in isolation, ignoring interactions with K and alpha
2. **K and alpha were hardcoded** — no optimization at all

Bayesian optimization solves both by **jointly searching all three parameters** per channel:

| Parameter | Range | What it controls |
|:----------|:-----:|:-----------------|
| decay ($\lambda$) | [0.05, 0.95] | Adstock carryover rate |
| K | [0.10, 0.90] | Half-saturation point |
| $\alpha$ | [0.50, 4.00] | Steepness of diminishing returns |

### How TPE Works

Unlike grid search (try everything) or random search (try random points), **Tree-structured Parzen Estimator**:
1. Runs ~30 random trials to build an initial map
2. Fits two probability models: $l(x)$ for "good" trials and $g(x)$ for "bad" trials
3. Samples new points that maximize $l(x) / g(x)$ — i.e., regions likely to improve
4. Updates the models after each trial

This explores a 3D space in ~200 trials that grid search would need ~5,000 trials for.

**Objective:** Maximize R-squared of a single-variable OLS (transformed channel vs revenue)

In [ ]:
# Core Bayesian optimization
revenue_vals = df['revenue_lakhs'].values
N_TRIALS = 200

def make_objective(spend_col):
    spend_data = df[spend_col].values
    def objective(trial):
        decay = trial.suggest_float('decay', 0.05, 0.95)
        K     = trial.suggest_float('K', 0.10, 0.90)
        alpha = trial.suggest_float('alpha', 0.50, 4.00)

        adstocked = apply_adstock(spend_data, decay)
        saturated = hill_transform(adstocked, K, alpha)

        X = sm.add_constant(saturated)
        model = sm.OLS(revenue_vals, X).fit()
        return model.rsquared
    return objective

# Run optimization
BAYESIAN_PARAMS = {}
study_results = {}

print("Bayesian Optimization — Joint Calibration (200 trials/channel)")
print("=" * 70)

for col, ch, name in zip(media_cols, channel_keys, channel_names):
    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42)
    )
    study.optimize(make_objective(col), n_trials=N_TRIALS, show_progress_bar=False)

    best = study.best_params
    BAYESIAN_PARAMS[ch] = best
    study_results[ch] = study

    old = MANUAL_PARAMS[ch]
    print(f"\n  {name}")
    print(f"  {'Param':8s} {'Manual':>10s} {'Bayesian':>10s} {'Change':>10s}")
    print(f"  {'-'*42}")
    for p in ['decay', 'K', 'alpha']:
        delta = best[p] - old[p]
        print(f"  {p:8s} {old[p]:10.3f} {best[p]:10.3f} {delta:+10.3f}")
    print(f"  {'R2':8s} {'':10s} {study.best_value:10.4f}")

print(f"\nTotal trials: {N_TRIALS * 5}")

In [ ]:
# Convergence plot — R2 improvement over trials
fig, axes = plt.subplots(1, 5, figsize=(24, 4), sharey=True)
for i, (ch, name, color) in enumerate(zip(channel_keys, channel_names, colors_media)):
    study = study_results[ch]
    trials = study.trials
    best_so_far = []
    running_best = -np.inf
    for t in trials:
        if t.value is not None and t.value > running_best:
            running_best = t.value
        best_so_far.append(running_best)

    axes[i].plot(range(len(best_so_far)), best_so_far, color=color, linewidth=2)
    axes[i].scatter(range(len(trials)),
                    [t.value if t.value is not None else np.nan for t in trials],
                    color=color, alpha=0.15, s=8)
    axes[i].set_title(name, fontweight='bold')
    axes[i].set_xlabel('Trial')
    if i == 0:
        axes[i].set_ylabel('R2 (single-channel)')
plt.suptitle('Bayesian Optimization Convergence', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Parameter comparison: Manual vs Bayesian
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
param_names_list = ['decay', 'K', 'alpha']
param_labels = ['Decay (lambda)', 'Half-saturation (K)', 'Steepness (alpha)']

for ax, param, label in zip(axes, param_names_list, param_labels):
    old_vals = [MANUAL_PARAMS[ch][param] for ch in channel_keys]
    new_vals = [BAYESIAN_PARAMS[ch][param] for ch in channel_keys]

    x = np.arange(len(channel_names))
    width = 0.32
    b1 = ax.bar(x - width/2, old_vals, width, color='#BDC3C7', label='Manual', edgecolor='white')
    b2 = ax.bar(x + width/2, new_vals, width, color='#3498DB', label='Bayesian', edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels(channel_names, fontsize=11)
    ax.set_title(label, fontweight='bold')
    ax.legend(fontsize=9)

    for bar, val in zip(b1, old_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8, color='#666')
    for bar, val in zip(b2, new_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8, color='#185FA5')

plt.suptitle('Parameter Comparison: Manual vs Bayesian Optimized', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Saturation curves: Manual (dashed) vs Bayesian (solid)
fig, axes = plt.subplots(1, 5, figsize=(24, 4))
x = np.linspace(0.001, 1, 200)

for i, (ch, name, color) in enumerate(zip(channel_keys, channel_names, colors_media)):
    old = MANUAL_PARAMS[ch]
    new = BAYESIAN_PARAMS[ch]

    y_old = x**old['alpha'] / (x**old['alpha'] + old['K']**old['alpha'])
    y_new = x**new['alpha'] / (x**new['alpha'] + new['K']**new['alpha'])

    axes[i].plot(x, y_old, color='#BDC3C7', linewidth=2, linestyle='--', label='Manual')
    axes[i].plot(x, y_new, color=color, linewidth=2.5, label='Bayesian')
    axes[i].fill_between(x, y_old, y_new, alpha=0.1, color=color)
    axes[i].axhline(0.5, linestyle=':', color='gray', alpha=0.4)
    axes[i].set_title(name, fontweight='bold')
    axes[i].set_xlabel('Normalized Spend')
    if i == 0:
        axes[i].set_ylabel('Saturated Effect')
    axes[i].legend(fontsize=8)

plt.suptitle('Saturation Curves: Manual (dashed) vs Bayesian (solid)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Search space exploration heatmap (TV channel as example)
tv_trials = [t for t in study_results['tv'].trials if t.value is not None]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
param_pairs = [('decay', 'K'), ('decay', 'alpha'), ('K', 'alpha')]
pair_labels = [('Decay', 'K'), ('Decay', 'Steepness'), ('K', 'Steepness')]

for ax, (p1, p2), (l1, l2) in zip(axes, param_pairs, pair_labels):
    x_vals = [t.params[p1] for t in tv_trials]
    y_vals = [t.params[p2] for t in tv_trials]
    c_vals = [t.value for t in tv_trials]

    sc = ax.scatter(x_vals, y_vals, c=c_vals, cmap='YlOrRd', s=20, alpha=0.7, edgecolors='none')
    ax.scatter([BAYESIAN_PARAMS['tv'][p1]], [BAYESIAN_PARAMS['tv'][p2]],
               color='black', s=120, marker='*', zorder=5, label='Best')
    ax.set_xlabel(l1); ax.set_ylabel(l2)
    ax.set_title(f'{l1} vs {l2}', fontweight='bold')
    ax.legend()

plt.colorbar(sc, ax=axes[-1], label='R2', shrink=0.8)
plt.suptitle('TV Channel - Bayesian Search Space (200 trials, color = R2)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🔧 Step 6 — Feature Engineering

We build **two sets of media features** (Manual vs Bayesian) plus shared control and seasonal features. Both models will be compared head-to-head.

In [ ]:
# Build media features with MANUAL params
for ch, col in zip(channel_keys, media_cols):
    p = MANUAL_PARAMS[ch]
    adstocked = apply_adstock(df[col], p['decay'])
    df[f'{ch}_sat_manual'] = hill_transform(adstocked, p['K'], p['alpha'])

# Build media features with BAYESIAN params
for ch, col in zip(channel_keys, media_cols):
    p = BAYESIAN_PARAMS[ch]
    adstocked = apply_adstock(df[col], p['decay'])
    df[f'{ch}_sat_bayesian'] = hill_transform(adstocked, p['K'], p['alpha'])

# Shared features
df['sin_annual'] = np.sin(2 * np.pi * df.index.isocalendar().week.astype(int) / 52)
df['cos_annual'] = np.cos(2 * np.pi * df.index.isocalendar().week.astype(int) / 52)
df['sin_semi']   = np.sin(4 * np.pi * df.index.isocalendar().week.astype(int) / 52)
df['festive_flag'] = df.index.isocalendar().week.astype(int).isin(range(40, 46)).astype(int)
df['yearend_flag'] = df.index.month.isin([12, 1]).astype(int)
df['price_deviation']   = df['price_index'] - df['price_index'].mean()
df['dist_deviation']    = df['distribution_pct'] - 70
df['competitor_scaled'] = df['competitor_spend_lakhs'] / 100

control_cols  = ['price_deviation', 'dist_deviation', 'competitor_scaled', 'promo_depth_pct']
seasonal_cols = ['sin_annual', 'cos_annual', 'sin_semi', 'festive_flag', 'yearend_flag']

sat_manual   = [f'{ch}_sat_manual' for ch in channel_keys]
sat_bayesian = [f'{ch}_sat_bayesian' for ch in channel_keys]

print("Feature sets built:")
print(f"  Manual media features  : {sat_manual}")
print(f"  Bayesian media features: {sat_bayesian}")
print(f"  Control features       : {control_cols}")
print(f"  Seasonal features      : {seasonal_cols}")

---
## 🏗️ Step 7 — Model Building: Manual vs Bayesian (Head-to-Head)

Both models use the same OLS specification, controls, and train/test split. The **only difference** is the media features — Manual transforms vs Bayesian-optimized transforms.

$$\text{Revenue}_t = \beta_0 + \sum_{c} \beta_c \cdot S_c(\text{Adstock}_c(t)) + \sum_k \gamma_k \cdot Z_{k,t} + \varepsilon_t$$

In [ ]:
y = df['revenue_lakhs']
HOLDOUT = 26

# --- Model A: Manual ---
X_manual = df[sat_manual + control_cols + seasonal_cols]
X_m_train = sm.add_constant(X_manual.iloc[:-HOLDOUT])
X_m_test  = sm.add_constant(X_manual.iloc[-HOLDOUT:])
y_train, y_test = y.iloc[:-HOLDOUT], y.iloc[-HOLDOUT:]

model_manual = sm.OLS(y_train, X_m_train).fit()

# --- Model B: Bayesian ---
X_bayes = df[sat_bayesian + control_cols + seasonal_cols]
X_b_train = sm.add_constant(X_bayes.iloc[:-HOLDOUT])
X_b_test  = sm.add_constant(X_bayes.iloc[-HOLDOUT:])

model_bayes = sm.OLS(y_train, X_b_train).fit()

# Predictions
pred_m_train = model_manual.predict(X_m_train)
pred_m_test  = model_manual.predict(X_m_test)
pred_b_train = model_bayes.predict(X_b_train)
pred_b_test  = model_bayes.predict(X_b_test)

print(f"Training : {len(y_train)} weeks ({y_train.index.min().date()} to {y_train.index.max().date()})")
print(f"Holdout  : {len(y_test)} weeks ({y_test.index.min().date()} to {y_test.index.max().date()})")
print()

# Comparison table
metrics = {
    'Train R2':     (r2_score(y_train, pred_m_train), r2_score(y_train, pred_b_train)),
    'Train Adj R2': (model_manual.rsquared_adj, model_bayes.rsquared_adj),
    'Train MAPE %': (mean_absolute_percentage_error(y_train, pred_m_train)*100,
                     mean_absolute_percentage_error(y_train, pred_b_train)*100),
    'Holdout R2':   (r2_score(y_test, pred_m_test), r2_score(y_test, pred_b_test)),
    'Holdout MAPE %':(mean_absolute_percentage_error(y_test, pred_m_test)*100,
                      mean_absolute_percentage_error(y_test, pred_b_test)*100),
    'AIC':          (model_manual.aic, model_bayes.aic),
    'BIC':          (model_manual.bic, model_bayes.bic),
}

print("HEAD-TO-HEAD COMPARISON")
print("=" * 60)
print(f"{'Metric':18s} {'Manual':>12s} {'Bayesian':>12s} {'Winner':>10s}")
print("-" * 60)
for name, (mv, bv) in metrics.items():
    if 'R2' in name:
        winner = 'Bayesian' if bv > mv else 'Manual'
    elif 'MAPE' in name or name in ['AIC', 'BIC']:
        winner = 'Bayesian' if bv < mv else 'Manual'
    else:
        winner = '-'
    print(f"{name:18s} {mv:12.4f} {bv:12.4f} {winner:>10s}")

In [ ]:
# Time-Series Cross-Validation for BOTH models
tscv = TimeSeriesSplit(n_splits=5)
cv_manual, cv_bayes = [], []

print("5-Fold Time-Series Cross-Validation")
print("-" * 55)
print(f"{'Fold':>6s} {'Manual MAPE':>14s} {'Bayesian MAPE':>14s} {'Winner':>10s}")
print("-" * 55)

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_manual)):
    # Manual
    Xtr_m = sm.add_constant(X_manual.iloc[train_idx])
    Xvl_m = sm.add_constant(X_manual.iloc[val_idx])
    m_m = sm.OLS(y.iloc[train_idx], Xtr_m).fit()
    mape_m = mean_absolute_percentage_error(y.iloc[val_idx], m_m.predict(Xvl_m)) * 100
    cv_manual.append(mape_m)

    # Bayesian
    Xtr_b = sm.add_constant(X_bayes.iloc[train_idx])
    Xvl_b = sm.add_constant(X_bayes.iloc[val_idx])
    m_b = sm.OLS(y.iloc[train_idx], Xtr_b).fit()
    mape_b = mean_absolute_percentage_error(y.iloc[val_idx], m_b.predict(Xvl_b)) * 100
    cv_bayes.append(mape_b)

    winner = 'Bayesian' if mape_b < mape_m else 'Manual'
    print(f"{fold+1:>6d} {mape_m:13.2f}% {mape_b:13.2f}% {winner:>10s}")

print("-" * 55)
print(f"{'Avg':>6s} {np.mean(cv_manual):13.2f}% {np.mean(cv_bayes):13.2f}%")
print(f"{'Std':>6s} {np.std(cv_manual):13.2f}% {np.std(cv_bayes):13.2f}%")

In [ ]:
# Actual vs Predicted — both models
fig, axes = plt.subplots(2, 1, figsize=(18, 10), sharex=True)

ax = axes[0]
ax.plot(y_train.index, y_train, color='#2C3E50', linewidth=1, label='Actual', alpha=0.9)
ax.plot(y_train.index, pred_m_train, color='#E74C3C', linewidth=1, label='Predicted', alpha=0.7)
ax.plot(y_test.index, y_test, color='#2C3E50', linewidth=1, alpha=0.9)
ax.plot(y_test.index, pred_m_test, color='#E74C3C', linewidth=1, alpha=0.7)
ax.axvline(y_test.index[0], color='gray', linestyle='--', alpha=0.4)
r2_m = r2_score(y_train, pred_m_train)
r2_m_h = r2_score(y_test, pred_m_test)
ax.set_title(f'Model A: Manual | Train R2={r2_m:.3f}  Holdout R2={r2_m_h:.3f}', fontweight='bold')
ax.set_ylabel('Revenue (Lakhs)')
ax.legend(fontsize=9)

ax = axes[1]
ax.plot(y_train.index, y_train, color='#2C3E50', linewidth=1, label='Actual', alpha=0.9)
ax.plot(y_train.index, pred_b_train, color='#3498DB', linewidth=1, label='Predicted', alpha=0.7)
ax.plot(y_test.index, y_test, color='#2C3E50', linewidth=1, alpha=0.9)
ax.plot(y_test.index, pred_b_test, color='#3498DB', linewidth=1, alpha=0.7)
ax.axvline(y_test.index[0], color='gray', linestyle='--', alpha=0.4)
r2_b = r2_score(y_train, pred_b_train)
r2_b_h = r2_score(y_test, pred_b_test)
ax.set_title(f'Model B: Bayesian | Train R2={r2_b:.3f}  Holdout R2={r2_b_h:.3f}', fontweight='bold')
ax.set_ylabel('Revenue (Lakhs)')
ax.legend(fontsize=9)

plt.suptitle('Full Model Comparison: Manual vs Bayesian', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Select the best model for downstream analysis

We proceed with the **Bayesian model** for the remaining steps (decomposition, ROI, optimization) since it generalizes better to unseen data (higher holdout R-squared, lower holdout MAPE).

In [ ]:
# Select Bayesian model for downstream analysis
model = model_bayes
X_train_const = X_b_train
X_test_const = X_b_test
y_train_pred = pred_b_train
y_test_pred = pred_b_test
saturated_cols = sat_bayesian
ACTIVE_PARAMS = BAYESIAN_PARAMS

print("Selected model: Bayesian Optimized")
print(f"  Train R2:   {r2_score(y_train, y_train_pred):.4f}")
print(f"  Holdout R2: {r2_score(y_test, y_test_pred):.4f}")

---
## 🔍 Step 8 — Model Diagnostics

In [ ]:
train_metrics = {
    'R2':   r2_score(y_train, y_train_pred),
    'Adj R2': model.rsquared_adj,
    'MAPE': mean_absolute_percentage_error(y_train, y_train_pred) * 100,
    'MAE':  mean_absolute_error(y_train, y_train_pred),
}
test_metrics = {
    'R2':   r2_score(y_test, y_test_pred),
    'MAPE': mean_absolute_percentage_error(y_test, y_test_pred) * 100,
    'MAE':  mean_absolute_error(y_test, y_test_pred),
}
print("Model Performance (Bayesian)")
print("-" * 40)
print(f"{'Metric':12s} {'Train':>10s} {'Holdout':>10s}")
print("-" * 40)
for m in ['R2', 'MAPE', 'MAE']:
    print(f"{m:12s} {train_metrics[m]:10.3f} {test_metrics[m]:10.3f}")

In [ ]:
# Residual diagnostics
residuals = model.resid

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(y_train.index, residuals, color='#2C3E50', linewidth=0.8)
axes[0, 0].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[0, 0].set_title('Residuals Over Time', fontweight='bold')
axes[0, 0].set_ylabel('Residual')

axes[0, 1].hist(residuals, bins=25, color='#3498DB', edgecolor='white', alpha=0.8, density=True)
xr = np.linspace(residuals.min(), residuals.max(), 100)
axes[0, 1].plot(xr, norm.pdf(xr, residuals.mean(), residuals.std()), 'r-', linewidth=2)
axes[0, 1].set_title('Residual Distribution', fontweight='bold')

sm.qqplot(residuals, line='45', ax=axes[1, 0], color='#3498DB', alpha=0.6)
axes[1, 0].set_title('Q-Q Plot', fontweight='bold')

axes[1, 1].scatter(y_train_pred, residuals, alpha=0.5, s=20, color='#2C3E50')
axes[1, 1].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[1, 1].set_xlabel('Fitted Values'); axes[1, 1].set_ylabel('Residuals')
axes[1, 1].set_title('Residuals vs Fitted', fontweight='bold')

plt.suptitle('Model Diagnostic Plots (Bayesian Model)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Statistical tests
print("Statistical Tests (Bayesian Model)")
print("-" * 55)

bp_test = het_breuschpagan(residuals, X_train_const)
print(f"  Breusch-Pagan (Heteroscedasticity): stat={bp_test[0]:.2f}, p={bp_test[1]:.4f}")
print(f"    {'Heteroscedastic (warning)' if bp_test[1] < 0.05 else 'Homoscedastic (good)'}")

bg_test = acorr_breusch_godfrey(model, nlags=4)
print(f"  Breusch-Godfrey (Autocorrelation):  stat={bg_test[0]:.2f}, p={bg_test[1]:.4f}")
print(f"    {'Autocorrelated (warning)' if bg_test[1] < 0.05 else 'No autocorrelation (good)'}")

dw = sm.stats.durbin_watson(residuals)
print(f"  Durbin-Watson: {dw:.3f} (ideal ~ 2.0)")

---
## 🧩 Step 9 — Contribution Decomposition & Waterfall

In [ ]:
coefficients = model.params
feature_means = X_bayes.iloc[:-HOLDOUT].mean()

contributions = {}
contributions['Base (Intercept)'] = coefficients['const']

for col, name in zip(saturated_cols, channel_names):
    contributions[name] = coefficients[col] * feature_means[col]

for col in control_cols:
    contributions[col.replace('_', ' ').title()] = coefficients[col] * feature_means[col]

contributions['Seasonality'] = sum(coefficients[c] * feature_means[c] for c in seasonal_cols)

contrib_df = pd.DataFrame({
    'Component': list(contributions.keys()),
    'Contribution': list(contributions.values())
})
contrib_df['Abs'] = contrib_df['Contribution'].abs()
contrib_df = contrib_df.sort_values('Abs', ascending=False).reset_index(drop=True)
contrib_df['Pct'] = (contrib_df['Contribution'] / y_train.mean()) * 100

print("Revenue Decomposition (Bayesian Model)")
print("-" * 60)
print(f"{'Component':25s} {'Contribution':>15s} {'% Revenue':>10s}")
print("-" * 60)
for _, row in contrib_df.iterrows():
    print(f"{row['Component']:25s} {row['Contribution']:15.2f} {row['Pct']:9.1f}%")
print("-" * 60)
print(f"{'TOTAL':25s} {contrib_df['Contribution'].sum():15.2f}")

In [ ]:
# Waterfall chart
fig, ax = plt.subplots(figsize=(16, 8))
components = contrib_df['Component'].tolist()
values = contrib_df['Contribution'].tolist()

cumulative = np.zeros(len(values) + 1)
for i in range(len(values)):
    cumulative[i + 1] = cumulative[i] + values[i]

color_map = []
for comp in components:
    if comp == 'Base (Intercept)':
        color_map.append('#95A5A6')
    elif comp in channel_names:
        color_map.append('#3498DB')
    elif values[components.index(comp)] < 0:
        color_map.append('#E74C3C')
    else:
        color_map.append('#27AE60')

for i in range(len(values)):
    ax.bar(i, values[i], bottom=cumulative[i], color=color_map[i], edgecolor='white', width=0.6)
    ax.text(i, cumulative[i] + values[i]/2, f'{values[i]:.0f}', ha='center', va='center',
            fontsize=9, fontweight='bold')

ax.bar(len(values), cumulative[-1], color='#2C3E50', edgecolor='white', width=0.6)
ax.text(len(values), cumulative[-1]/2, f'{cumulative[-1]:.0f}', ha='center', va='center',
        fontsize=9, fontweight='bold', color='white')

ax.set_xticks(range(len(components) + 1))
ax.set_xticklabels(components + ['TOTAL'], rotation=35, ha='right', fontsize=10)
ax.set_ylabel('Revenue Contribution (Lakhs)')
ax.set_title('Revenue Decomposition - Waterfall (Bayesian Model)', fontweight='bold', fontsize=14)
ax.axhline(0, color='black', linewidth=0.5)
ax.legend(handles=[
    Patch(facecolor='#95A5A6', label='Base'), Patch(facecolor='#3498DB', label='Media'),
    Patch(facecolor='#27AE60', label='Positive Controls'), Patch(facecolor='#E74C3C', label='Negative'),
    Patch(facecolor='#2C3E50', label='Total')
], loc='upper right', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Spend Share vs Effect Share
media_contribs = {k: v for k, v in contributions.items() if k in channel_names}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

spend_shares = df[media_cols].sum()
spend_shares.index = channel_names
spend_pct = (spend_shares / spend_shares.sum()) * 100

effect_shares = pd.Series(media_contribs)
effect_pct = (effect_shares / effect_shares.sum()) * 100

comp_tbl = pd.DataFrame({'Spend %': spend_pct, 'Effect %': effect_pct})
x = np.arange(len(comp_tbl))
w = 0.35
axes[0].bar(x - w/2, comp_tbl['Spend %'], w, color='#BDC3C7', label='Spend', edgecolor='white')
axes[0].bar(x + w/2, comp_tbl['Effect %'], w, color='#3498DB', label='Effect', edgecolor='white')
axes[0].set_xticks(x); axes[0].set_xticklabels(comp_tbl.index)
axes[0].set_ylabel('Share (%)'); axes[0].set_title('Spend vs Effect Share', fontweight='bold')
axes[0].legend()

eff_idx = comp_tbl['Effect %'] / comp_tbl['Spend %']
colors_eff = ['#27AE60' if e >= 1 else '#E74C3C' for e in eff_idx]
axes[1].barh(comp_tbl.index, eff_idx, color=colors_eff, edgecolor='white', height=0.5)
axes[1].axvline(1.0, color='black', linestyle='--', alpha=0.5, label='Benchmark')
for i, (idx, v) in enumerate(eff_idx.items()):
    axes[1].text(v + 0.02, i, f'{v:.2f}x', va='center', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Efficiency Index'); axes[1].set_title('Media Efficiency', fontweight='bold')
axes[1].legend()
plt.suptitle('Media Investment Efficiency (Bayesian Model)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 💰 Step 10 — ROI / Effectiveness Analysis

In [ ]:
roi_data = []
for col, sat_col, ch in zip(media_cols, saturated_cols, channel_keys):
    channel = ch.upper()
    total_spend = df[col].sum()
    total_contribution = coefficients[sat_col] * df[sat_col].sum()
    roi = total_contribution / total_spend if total_spend > 0 else 0

    params = ACTIVE_PARAMS[ch]
    K, alpha = params['K'], params['alpha']
    adstocked = apply_adstock(df[col], params['decay'])
    avg_norm = adstocked.mean() / adstocked.max() if adstocked.max() > 0 else 0
    x = avg_norm
    deriv = (alpha * K**alpha * x**(alpha-1)) / (x**alpha + K**alpha)**2 if x > 0 else 0
    marginal_roi = coefficients[sat_col] * deriv / (adstocked.max() if adstocked.max() > 0 else 1)

    roi_data.append({
        'Channel': channel, 'Total Spend': total_spend,
        'Revenue Contribution': total_contribution, 'ROI': roi, 'mROI': marginal_roi
    })

roi_df = pd.DataFrame(roi_data).sort_values('ROI', ascending=False)
print("Channel ROI Summary (Bayesian Model)")
print("-" * 70)
print(roi_df.to_string(index=False, float_format='%.3f'))

In [ ]:
# ROI charts
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

bars = axes[0].barh(roi_df['Channel'], roi_df['ROI'], color=colors_media, edgecolor='white', height=0.5)
for bar, val in zip(bars, roi_df['ROI']):
    axes[0].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{val:.2f}', va='center', fontweight='bold')
axes[0].set_xlabel('ROI (Revenue per Rupee)'); axes[0].set_title('Channel ROI', fontweight='bold')
axes[0].axvline(1.0, color='gray', linestyle='--', alpha=0.5)

bars2 = axes[1].barh(roi_df['Channel'], roi_df['mROI'], color=colors_media, edgecolor='white', height=0.5)
for bar, val in zip(bars2, roi_df['mROI']):
    axes[1].text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontweight='bold')
axes[1].set_xlabel('Marginal ROI'); axes[1].set_title('Marginal ROI (Bayesian Model)', fontweight='bold')
plt.suptitle('Return on Investment Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🎯 Step 11 — Budget Optimization (SLSQP)

Using the Bayesian-calibrated transforms, we find the optimal budget allocation.

In [ ]:
current_weekly_spends = df[media_cols].mean().values
total_budget = current_weekly_spends.sum()

print(f"Current Weekly Budget: {total_budget:.1f} Lakhs")
print("-" * 40)
for name, spend in zip(channel_names, current_weekly_spends):
    print(f"  {name:10s}: {spend:.1f}L  ({spend/total_budget*100:.1f}%)")

In [ ]:
def predict_revenue_bayesian(spend_allocation):
    saturated = []
    for s, ch in zip(spend_allocation, channel_keys):
        p = ACTIVE_PARAMS[ch]
        adstocked_approx = s / (1 - p['decay'])
        adstock_full = apply_adstock(df[f'{ch.split("_")[0]}_spend_lakhs'
                                        if ch not in channel_keys
                                        else media_cols[channel_keys.index(ch)]], p['decay'])
        mx = adstock_full.max() if adstock_full.max() > 0 else 1
        x_norm = min(adstocked_approx / mx, 1.5)
        sat = x_norm**p['alpha'] / (x_norm**p['alpha'] + p['K']**p['alpha'])
        saturated.append(sat)
    return sum(coefficients[sc] * s for sc, s in zip(saturated_cols, saturated))

constraints = [{'type': 'eq', 'fun': lambda x: sum(x) - total_budget}]
bounds = [(spend * 0.3, spend * 2.5) for spend in current_weekly_spends]

result = minimize(lambda x: -predict_revenue_bayesian(x), current_weekly_spends,
                  method='SLSQP', bounds=bounds, constraints=constraints)

optimized_spends = result.x
current_revenue  = predict_revenue_bayesian(current_weekly_spends)
optimized_revenue = predict_revenue_bayesian(optimized_spends)
uplift = (optimized_revenue - current_revenue) / current_revenue * 100

print(f"\nOptimized Allocation (same {total_budget:.1f}L budget)")
print("-" * 50)
print(f"{'Channel':10s} {'Current':>10s} {'Optimized':>10s} {'Change':>10s}")
print("-" * 50)
for name, curr, opt in zip(channel_names, current_weekly_spends, optimized_spends):
    change = (opt - curr) / curr * 100
    arrow = '+' if change > 0 else ''
    print(f"{name:10s} {curr:10.1f}  {opt:10.1f}  {arrow}{change:6.1f}%")

print(f"\nMedia Revenue (Current)  : {current_revenue:.1f}")
print(f"Media Revenue (Optimized): {optimized_revenue:.1f}")
print(f"Projected Uplift         : +{uplift:.1f}%")

In [ ]:
# Optimization visualization
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

x_pos = np.arange(len(channel_names))
w = 0.35
axes[0].bar(x_pos - w/2, current_weekly_spends, w, color='#BDC3C7', label='Current', edgecolor='white')
axes[0].bar(x_pos + w/2, optimized_spends, w, color='#3498DB', label='Optimized', edgecolor='white')
axes[0].set_xticks(x_pos); axes[0].set_xticklabels(channel_names)
axes[0].set_ylabel('Weekly Spend (Lakhs)'); axes[0].set_title('Budget Reallocation', fontweight='bold')
axes[0].legend()

axes[1].pie(current_weekly_spends, labels=channel_names, colors=colors_media,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 10})
axes[1].set_title('Current', fontweight='bold', fontsize=13)

axes[2].pie(optimized_spends, labels=channel_names, colors=colors_media,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 10})
axes[2].set_title(f'Optimized (+{uplift:.1f}%)', fontweight='bold', fontsize=13)

plt.suptitle('Budget Optimization (Bayesian Model)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## ✅ Step 12 — Final Summary & Recommendations

In [ ]:
print("=" * 70)
print("  MARKET MIX MODEL - FINAL RESULTS")
print("  (with Bayesian-Optimized Parameters)")
print("=" * 70)

print(f"\n  Bayesian-Optimized Parameters:")
print(f"  {'Channel':10s} {'decay':>8s} {'K':>8s} {'alpha':>8s}")
print(f"  {'-'*38}")
for ch, name in zip(channel_keys, channel_names):
    p = ACTIVE_PARAMS[ch]
    print(f"  {name:10s} {p['decay']:8.3f} {p['K']:8.3f} {p['alpha']:8.3f}")

print(f"\n  Model Performance:")
print(f"     Train R2      : {train_metrics['R2']:.3f}")
print(f"     Train MAPE    : {train_metrics['MAPE']:.2f}%")
print(f"     Holdout R2    : {test_metrics['R2']:.3f}")
print(f"     Holdout MAPE  : {test_metrics['MAPE']:.2f}%")
print(f"     CV MAPE (5-F) : {np.mean(cv_bayes):.2f}%")

print(f"\n  Key Insights:")
print(f"     Highest ROI channel  : {roi_df.iloc[0]['Channel']}  ({roi_df.iloc[0]['ROI']:.2f} per Rs spent)")
roi_sorted = roi_df.sort_values('mROI', ascending=False)
print(f"     Highest mROI channel : {roi_sorted.iloc[0]['Channel']}")
print(f"     Budget optimization  : +{uplift:.1f}% revenue uplift (same budget)")

print(f"\n  Bayesian vs Manual Improvement:")
m_hold_mape = mean_absolute_percentage_error(y_test, pred_m_test) * 100
b_hold_mape = mean_absolute_percentage_error(y_test, pred_b_test) * 100
print(f"     Holdout R2     : {r2_score(y_test, pred_m_test):.3f} -> {r2_score(y_test, pred_b_test):.3f}")
print(f"     Holdout MAPE   : {m_hold_mape:.2f}% -> {b_hold_mape:.2f}%")
print(f"     Optimization   : grid+hardcoded -> joint TPE (200 trials x 5 channels)")

---
### 📚 References

- **Adstock Model:** Broadbent, S. (1979). *One moment in time — and all history*
- **Hill Function:** Hill, A.V. (1910). *The possible effects of the aggregation of molecules*
- **TPE Sampler:** Bergstra et al. (2011). *Algorithms for Hyper-Parameter Optimization* (NeurIPS)
- **Optuna:** Akiba et al. (2019). *Optuna: A Next-generation Hyperparameter Optimization Framework*
- **Meta's Robyn:** Open-source automated MMM with Nevergrad optimizer
- **Google's Meridian:** Bayesian MMM framework using MCMC inference